# Kurdish Wiktionary starter: the dataset in 5 minutes

Load the lexicon, look up a word, listen to a recording, and see what the data covers.

Text is CC BY-SA 4.0 from [Wîkîferheng](https://ku.wiktionary.org). Media licences vary per file: see the `media` table.

In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd

# On Kaggle the data is under /kaggle/input. Elsewhere, set WIKTIONARY_DATA to the release folder.
ROOT = Path(os.environ.get("WIKTIONARY_DATA", "/kaggle/input"))


def find(name):
    """Locate a file wherever the upload placed it."""
    return Path(glob.glob(f"{ROOT}/**/{name}", recursive=True)[0])


def table(name, **kwargs):
    return pd.read_parquet(find(f"{name}.parquet"), **kwargs)


entries = table("entries")
print(f"{len(entries):,} entries in {entries.language_code.nunique():,} language codes")
entries.head()

## Choose the right subset

This is the *Kurdish-language* Wiktionary: it describes words of many languages in Kurdish. Most users want to filter by language.
Bot-made entries are flagged, because the wiki warns that they may contain translation errors.

In [ ]:
kurmanci = entries[entries.language_code == "ku"]
handmade = entries[entries.is_auto_generated == 0]
print(f"Kurmancî entries: {len(kurmanci):,}")
print(f"Not bot-made:     {len(handmade):,}")
entries.groupby(["language_code", "language_name"]).size().sort_values(ascending=False).head(15)

## Look up a word

Every section of a wiki entry has its own table, joined on `entry_id`.

In [ ]:
senses = table("senses", columns=["entry_id", "pos", "ordinal", "depth", "labels", "definition"])
pronunciations = table("pronunciations")
translations = table("translations", columns=["entry_id", "target_language", "term"])
relations = table("relations", columns=["entry_id", "relation_type", "term"])


def lookup(word, language="ku"):
    match = entries[(entries.title == word) & (entries.language_code == language)]
    if match.empty:
        print("Not found")
        return None
    entry_id = match.entry_id.iloc[0]
    print("Definitions")
    for row in senses[senses.entry_id == entry_id].itertuples():
        print(f"  {'  ' * row.depth}{row.ordinal}. ({row.pos}) {row.definition}")
    print("Pronunciation:", pronunciations[(pronunciations.entry_id == entry_id) & (pronunciations.kind == "ipa")].value.tolist())
    print("Synonyms:", relations[(relations.entry_id == entry_id) & (relations.relation_type == "synonyms")].term.tolist())
    print("Translations:", len(translations[translations.entry_id == entry_id]))
    return entry_id


entry_id = lookup("heval")

## Listen to a pronunciation

Recordings are in `media/audio.zip`. The `media` table gives each file's path, licence and author.

In [ ]:
import tempfile
import zipfile

from IPython.display import Audio, display

media = table("media")
clips = pronunciations[(pronunciations.entry_id == entry_id) & (pronunciations.kind == "audio")]
clip = media[media.media_id.isin(clips.media_id) & (media.status == "present")].iloc[0]
print(clip.commons_file, "|", clip.license, "|", clip.artist)

archive = find("audio.zip")
with zipfile.ZipFile(archive) as bundle:
    path = bundle.extract(clip.local_path, tempfile.mkdtemp())
display(Audio(path))

## What does the lexicon cover?

In [ ]:
import matplotlib.pyplot as plt

counts = (
    senses[senses.entry_id.isin(kurmanci.entry_id) & ~senses.pos.str.endswith("_form")]
    .pos.value_counts()
    .head(10)
    .sort_values()
)
ax = counts.plot.barh(color="#0E8AA3", figsize=(8, 4))
ax.set_title("Kurmancî definitions by part of speech (top 10, inflected forms excluded)")
ax.set_xlabel("Definitions")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Query it with SQL

The whole dataset is also one SQLite file.

In [ ]:
import sqlite3

database = find("wiktionary.sqlite")
with sqlite3.connect(f"file:{database}?mode=ro", uri=True) as conn:
    result = pd.read_sql_query(
        """
        SELECT e.title, q.year, q.author, q.work, q.passage
        FROM quotations q JOIN entries e USING (entry_id)
        WHERE q.author = 'Ehmedê Xanî'
        LIMIT 5
        """,
        conn,
    )
result

## Next

- `DATA_DICTIONARY.md` explains every table and column.
- `QUALITY.md` lists coverage and known limits.
- `mdict/` holds a ready-made offline dictionary for GoldenDict and MDict.